## Content Based Recommender

In [2]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import CountVectorizer

In [57]:
users_movies = pd.read_csv("users_movies.csv")

In [4]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

## Choose Genre as Content feature

In [6]:
train['genres']

0         Comedy|Fantasy|Romance|Sci-Fi
1                      Mystery|Thriller
2                 Drama|Sci-Fi|Thriller
3                                Action
4                        Comedy|Mystery
                      ...              
800162                     Comedy|Drama
800163           Action|Sci-Fi|Thriller
800164                           Comedy
800165           Comedy|Musical|Romance
800166                     Action|Drama
Name: genres, Length: 800167, dtype: object

In [8]:
genres = set()
train['genres'] = train['genres'].str.replace('|', ' ')
for i in train['genres']:
    list_i = i.split()
    for j in list_i:
        genres.add(j)
print(genres)

{'Drama', 'Musical', 'Film-Noir', 'Thriller', 'Comedy', 'Horror', 'Documentary', 'War', "Children's", 'Fantasy', 'Action', 'Crime', 'Romance', 'Mystery', 'Sci-Fi', 'Western', 'Animation', 'Adventure'}


In [10]:
movies = train[['movie_id', 'title', 'genres']].drop_duplicates(subset='movie_id')

In [12]:
movies

,movie_id,title,genres
0,788,"Nutty Professor, The (1996)",Comedy Fantasy Romance Sci-Fi
1,1092,Basic Instinct (1992),Mystery Thriller
2,1653,Gattaca (1997),Drama Sci-Fi Thriller
3,544,Striking Distance (1993),Action
4,492,Manhattan Murder Mystery (1993),Comedy Mystery
...,...,...,...
787557,3485,Autopsy (Macchie Solari) (1975),Horror
788721,1820,"Proposition, The (1998)",Drama
792133,884,Sweet Nothing (1995),Drama
792421,3888,Skipped Parts (2000),Drama Romance


## Create Genre Matrix

In [14]:
for i in genres:
    movies[i] = movies.genres.apply(lambda x: int(i in x))
movie_genres = movies.set_index('movie_id').drop(columns=[ 'title', 'genres'])
movie_genres


,Drama,Musical,Film-Noir,Thriller,Comedy,Horror,Documentary,War,Children's,Fantasy,Action,Crime,Romance,Mystery,Sci-Fi,Western,Animation,Adventure
movie_id,,,,,,,,,,,,,,,,,,
788,0,0,0,0,1,0,0,0,0,1,0,0,1,0,1,0,0,0
1092,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0
1653,1,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0
544,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0
492,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3485,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0
1820,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
884,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [16]:
seen = train.groupby("user_id")["movie_id"].apply(set).to_dict()
liked = (train[train["rating"] >= 4].groupby("user_id")["movie_id"].apply(list).to_dict())
test_set = (test[test["rating"] >= 4].groupby("user_id")["movie_id"].apply(list).to_dict())
watched = train.groupby('user_id')['movie_id']
watched = watched.apply(set).to_dict()

## Recommend movies to user based on Cosine Similarity from values in genre matrix

In [18]:
def content(user_id):
    s = seen.get(user_id, [])
    l  = liked.get(user_id, [])
    l_m = []
    
    for i in l:
        if i in movie_genres.index:
            l_m.append(i)
    if len(l_m) == 0:
        return {}
        
    vex = movie_genres.loc[l_m]
    profile = vex.mean(axis=0)
    profile = profile.values.reshape(1, -1)
    vals = movie_genres.values
    sr = cosine_similarity(profile, vals)[0]

    ids = movie_genres.index.tolist()
    m_sr = dict(zip(ids, sr))
    w_m = watched.get(user, set())
    
    for i in w_m:
        m_sr.pop(i, None)
    return m_sr

## Example recommendations for User 1

In [59]:
def map_ids_to_titles(movie_ids, movies, k=10):
    sr_sort = sorted(movie_ids, key=movie_ids.get, reverse=True)
    movie_ids = sr_sort[:k]
    movie_map = dict(zip(movies["movie_id"], movies["title"]))
    return [movie_map.get(mid, "Unknown") for mid in movie_ids]

movie_ids = content(1)
print("Top 10 movie recommendations for User 1\n")
print(map_ids_to_titles(movie_ids, users_movies))

Top 10 movie recommendations for User 1

['Watership Down (1978)', 'Wizard of Oz, The (1939)', 'Secret Garden, The (1993)', 'Old Yeller (1957)', 'Little Princess, A (1995)', 'Parent Trap, The (1998)', 'Parent Trap, The (1961)', 'Little Princess, The (1939)', 'So Dear to My Heart (1949)', 'Prancer (1989)']


In [20]:
def ndcg_at_k(recommended, relevant, k=10):
    recommended = recommended[:k]
    relevant = set(relevant)
    dcg = 0
    for i, item in enumerate(recommended):
        if item in relevant:
            dcg += 1 / np.log2(i + 2)

    ideal_hits = min(len(relevant), k)
    idcg = sum(1 / np.log2(i + 2) for i in range(ideal_hits))
    return dcg / idcg if idcg > 0 else 0


## Test Recommender for top 10 and top 100 recommendations for each user in test set

In [26]:
total_hits = 0
total_relevant = 0
precision_sum = 0
recall_sum = 0
ndcg_sum = 0
num_users = 0

for user, movies in test_set.items():
    if user not in liked:
        continue

    m_sr = content(user)
    sr_sort = sorted(m_sr, key=m_sr.get, reverse=True)
    top_10 = sr_sort[:10]

    movie_set = set(movies)
    hits = len(movie_set.intersection(top_10))

    precision = hits / 10
    recall = hits / len(movie_set) if len(movie_set) > 0 else 0
    ndcg = ndcg_at_k(top_10, movie_set, k=10)

    total_hits += hits
    total_relevant += len(movie_set)
    precision_sum += precision
    recall_sum += recall
    ndcg_sum += ndcg
    num_users += 1

avg_precision = precision_sum / num_users if num_users > 0 else 0
avg_recall = recall_sum / num_users if num_users > 0 else 0
avg_ndcg = ndcg_sum / num_users if num_users > 0 else 0
micro_recall = total_hits / total_relevant if total_relevant > 0 else 0

print("Users evaluated:", num_users)
print("Hits:", total_hits)
print("Possible hits:", total_relevant)
print(f"Precision@10: {avg_precision:.4f}")
print(f"Recall@10: {avg_recall:.4f}")
print(f"NDCG@10: {avg_ndcg:.4f}")
print(f"Micro Recall@10: {micro_recall:.4f}")

Users evaluated: 6015
Hits: 2324
Possible hits: 114951
Precision@10: 0.0386
Recall@10: 0.0281
NDCG@10: 0.0447
Micro Recall@10: 0.0202


In [62]:
total_hits = 0
total_relevant = 0
precision_sum = 0
recall_sum = 0
ndcg_sum = 0
num_users = 0

for user, movies in test_set.items():
    if user not in liked:
        continue

    m_sr = content(user)
    sr_sort = sorted(m_sr, key=m_sr.get, reverse=True)
    top_100 = sr_sort[:100]

    movie_set = set(movies)
    hits = len(movie_set.intersection(top_100))

    precision = hits / 100
    recall = hits / len(movie_set) if len(movie_set) > 0 else 0
    ndcg = ndcg_at_k(top_100, movie_set, k=100)

    total_hits += hits
    total_relevant += len(movie_set)
    precision_sum += precision
    recall_sum += recall
    ndcg_sum += ndcg
    num_users += 1

avg_precision = precision_sum / num_users if num_users > 0 else 0
avg_recall = recall_sum / num_users if num_users > 0 else 0
avg_ndcg = ndcg_sum / num_users if num_users > 0 else 0
micro_recall = total_hits / total_relevant if total_relevant > 0 else 0

print("Users evaluated:", num_users)
print("Hits:", total_hits)
print("Possible hits:", total_relevant)
print(f"Precision@100: {avg_precision:.4f}")
print(f"Recall@100: {avg_recall:.4f}")
print(f"NDCG@100: {avg_ndcg:.4f}")
print(f"Micro Recall@100: {micro_recall:.4f}")

Users evaluated: 6015
Hits: 13380
Possible hits: 114951
Precision@100: 0.0222
Recall@100: 0.1496
NDCG@100: 0.0833
Micro Recall@100: 0.1164
